
# From RNNs to Transformers

## The Problem with Sequence Data

Standard Neural Networks (MLPs) expect a fixed-size input. But language is dynamic:

-   "Hello" (1 word)
-   "The quick brown fox&#x2026;" (5 words)

We need models that can handle variable-length sequences and "remember" context.

## Recurrent Neural Networks (RNNs)

The RNN was the first solution. It processes words one by one, maintaining a **Hidden State** (memory).

-   **Step 1**: Read "The". Update Memory.
-   **Step 2**: Read "quick". Update Memory based on "quick" + "Old Memory".

$$h_t = \tanh(W x_t + U h_{t-1})$$

### The Bottleneck

1.  **Sequential**: You cannot process the 100th word until you calculate the 99th. This makes training very slow (cannot parallelize).
2.  **Vanishing Memory**: By the time the RNN reaches the end of a long paragraph, it has forgotten the beginning.

**LSTMs (Long Short-Term Memory)** improved this with "Gates" (Forget Gate, Input Gate) to keep memory longer, but they were still slow.

### Visualizing Sequential Processing

Let's see how the hidden state evolves as an RNN reads a sentence.

In [ ]:
import torch
import torch.nn as nn

# A simple RNN cell
rnn_cell = nn.RNNCell(input_size=4, hidden_size=4)

# Simulate 3 word embeddings (simplified)
words = ["The", "cat", "sat"]
embeddings = [torch.randn(1, 4) for _ in words]

# Process sequentially
hidden = torch.zeros(1, 4)  # Initial memory (blank slate)

print("RNN Processing (Each step waits for the previous):")
for i, (word, emb) in enumerate(zip(words, embeddings)):
    hidden = rnn_cell(emb, hidden)  # Update memory
    print(f"  Step {i+1}: Read '{word}' -> Hidden: {hidden[0,:2].tolist()}")

**Key Point**: Notice how each step **must wait** for the previous hidden state. This is the sequential bottleneck.

## The Attention Mechanism

In 2014, researchers asked: **"Instead of trying to squash the whole sentence into one tiny memory vector, why not let the model 'look back' at any word it wants?"**

This is **Attention**. When predicting the next word, the model assigns a "weight" to every previous word.

-   **"I poured cereal into the&#x2026;"** -> Focus on "cereal" and "poured". Predict: "bowl".

## The Transformer (2017)

The paper **"Attention is All You Need"** changed everything. It proved we don't need recurrence (RNNs) at all. We can process the **entire sentence at once** using **Self-Attention**.

| Feature        | RNN/LSTM             | Transformer                      |
|-------------- |-------------------- |-------------------------------- |
| **Processing** | Sequential (Slow)    | Parallel (Fast)                  |
| **Context**    | Local (Nearby words) | Global (All words see all words) |
| **Distance**   | Linearly far         | Constant (1 step away)           |

### The Architecture

1.  **Encoder**: Reads the input text and creates a rich representation. (Used in **BERT**).
2.  **Decoder**: Generates output text step-by-step. (Used in **GPT**).
3.  **Positional Encoding**: Since Transformers process parallelly, they don't know word order. We inject "Position" numbers so "Dog bites Man" $\ne$ "Man bites Dog".
4.  **Feed-Forward Network (FFN)**: After attention, each position passes through a small MLP independently. This adds non-linearity and increases model capacity.

## Practical Demonstration: Self-Attention in PyTorch

We will implement the mathematical core of the Transformer: **Scaled Dot-Product Attention**.

$$Attention(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

-   **Query (Q)**: What I am looking for?
-   **Key (K)**: What can I offer?
-   **Value (V)**: What information do I actually hold?

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# 1. Setup Dummy Data
# A sentence with 3 words, each represented by a vector of size 4
batch_size = 1
seq_len = 3   # "I love AI"
d_model = 4   # Embedding Dimension

torch.manual_seed(42)
x = torch.randn(batch_size, seq_len, d_model)

print(f"Input Shape: {x.shape}")

# 2. Define Projection Matrices (Wq, Wk, Wv)
# In a real model, these are learned weights
W_q = torch.nn.Linear(d_model, d_model)
W_k = torch.nn.Linear(d_model, d_model)
W_v = torch.nn.Linear(d_model, d_model)

# 3. Create Q, K, V
Q = W_q(x)
K = W_k(x)
V = W_v(x)

# 4. Calculate Attention Scores (Q * K^T)
# transpose(-2, -1) swaps the last two dimensions for matrix multiplication
scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(d_model)

print(f"Raw Scores (Word vs Word):\n{scores[0].detach().numpy()}")

# 5. Softmax (Turn into probabilities)
attention_weights = F.softmax(scores, dim=-1)

print(f"\nAttention Weights (Sum to 1):\n{attention_weights[0].detach().numpy()}")

# 6. Final Output (Weights * V)
output = torch.matmul(attention_weights, V)
print(f"\nOutput Shape: {output.shape}")

### Visualizing Attention

This heatmap shows how much each word "attends" to every other word.

In [ ]:
import seaborn as sns

plt.figure(figsize=(6, 5))
sns.heatmap(attention_weights[0].detach().numpy(), 
            xticklabels=["I", "Love", "AI"], 
            yticklabels=["I", "Love", "AI"], 
            annot=True, cmap='Blues')
plt.title("Self-Attention Weights")
plt.xlabel("Key (Source)")
plt.ylabel("Query (Target)")
plt.show()

### Causal Masking (For Decoders/GPT)

In generation tasks, a word should **not** see future words (that would be cheating). We use a **Causal Mask** to block the future.

In [ ]:
# Create a mask that blocks "future" positions
# Upper triangle filled with -infinity (becomes 0 after softmax)
seq_len = 3
mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1) * float('-inf')

print("Causal Mask (0 = can see, -inf = blocked):")
print(mask)

# Apply mask to attention scores
masked_scores = scores + mask
masked_weights = F.softmax(masked_scores, dim=-1)

print("\nMasked Attention Weights:")
print(masked_weights[0].detach().numpy())
# Word 1 sees only itself, Word 2 sees 1-2, Word 3 sees all

**Key Point**: This is why GPT models generate text one token at a time - each new token can only "see" previous tokens.

## Practical Demonstration: Using a Pretrained Transformer (Hugging Face)

We rarely build Transformers from scratch. We use libraries like `transformers`.

In [ ]:
from transformers import BertTokenizer, BertModel

# 1. Load Pretrained BERT
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

# 2. Tokenize Input
text = "Transformers are fast."
inputs = tokenizer(text, return_tensors="pt")
print(f"Token IDs: {inputs['input_ids']}")

# See what BERT actually sees (word pieces + special tokens)
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
print(f"Tokens: {tokens}")  # ['[CLS]', 'transformers', 'are', 'fast', '.', '[SEP]']

# 3. Forward Pass
with torch.no_grad():
    outputs = model(**inputs)

# The "Hidden States" representing the meaning of the sentence
# Shape: (Batch, Seq_Len, Hidden_Dim) -> (1, 6, 768)
last_hidden_states = outputs.last_hidden_state
print(f"Contextual Embeddings Shape: {last_hidden_states.shape}")

## Exercises

### Positional Encoding

Without this, the model treats "I eat" the same as "Eat I".

**Real Transformers** use sinusoidal functions that encode position smoothly: $$PE_{(pos, 2i)} = \sin(pos / 10000^{2i/d})$$ $$PE_{(pos, 2i+1)} = \cos(pos / 10000^{2i/d})$$

This allows the model to learn relative positions (e.g., "3 words apart") more easily than naive integers.

**Exercise**: Try a simplified version:

-   Create a random tensor.
-   Add a "positional vector" to it (e.g., adding 1 to the first word, 2 to the second).
-   Check how this changes the attention scores (Run the QK calculation again).

### Multi-Head Attention

The real power comes from having **multiple** sets of Q, K, V (Heads), allowing the model to focus on grammar AND meaning simultaneously.

-   Use `torch.nn.MultiheadAttention(embed_dim=4, num_heads=2)`.
-   Pass `x` as both query, key, and value.

## Summary

1.  **RNNs**: Sequential, slow, forgetful. Good for time-series, bad for long text.
2.  **Attention**: The ability to look at the whole sequence at once.
3.  **Transformer**: Uses Self-Attention + Positional Encoding to process text in parallel.
4.  **BERT/GPT**: Giant Transformers pre-trained on the internet.